# 01 — Audit initial des données SpamGuard-FR

Objectif : décrire le dataset et ses limites avant tout split ou entraînement. Le texte français est une **traduction automatique** du SMS Spam Collection anglais ; ce n'est pas un corpus de SMS natifs collectés en France. La fiche Hugging Face annonce une licence GPL, tandis que la source UCI (DOI `10.24432/C5CC84`) annonce CC BY 4.0.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'config' / 'config.yaml').exists())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

from spamguard.audit import build_audit_report, run_audit
from spamguard.data import load_processed_data, prepare_dataset
from spamguard.features import add_exploratory_features

RANDOM_SEED = 42
processed_path = prepare_dataset()
df = load_processed_data(processed_path)
features = add_exploratory_features(df)
print(f'Dataset préparé : {processed_path}')

Dataset préparé : data/processed/sms_fr.csv


## Dimensions générales

In [2]:
dimensions = pd.Series({'nombre de lignes': len(df), 'nombre de colonnes': df.shape[1]})
display(dimensions.to_frame('valeur'))
print('Labels uniques :', sorted(df['label'].unique()))
display(df.isna().sum().to_frame('valeurs manquantes'))
exact_duplicates = df.duplicated(['label', 'text_fr', 'text_en']).sum()
print(f'Doublons exacts (hors identifiant technique) : {exact_duplicates}')

,valeur
nombre de lignes,5572
nombre de colonnes,4


Labels uniques : ['ham', 'spam']


,valeurs manquantes
id,0
label,0
text_fr,0
text_en,0


Doublons exacts (hors identifiant technique) : 415


## Distribution des classes

In [3]:
class_counts = df['label'].value_counts().reindex(['ham', 'spam'])
class_distribution = pd.DataFrame({
    'nombre': class_counts,
    'pourcentage': (class_counts / len(df) * 100).round(2),
})
display(class_distribution)
imbalance_ratio = class_counts.max() / class_counts.min()
print(f'Ratio majorité/minorité : {imbalance_ratio:.3f}:1')
ax = class_counts.plot.bar(color=['#3b82f6', '#ef4444'], rot=0, title='Distribution des classes')
ax.set(xlabel='Label', ylabel='Nombre de messages')
plt.tight_layout()
plt.show()

,nombre,pourcentage
label,,
ham,4825,86.59
spam,747,13.41


Ratio majorité/minorité : 6.459:1


/var/folders/vl/wsdldpb93sv0d2b1g9kn49380000gn/T/ipykernel_16084/799221134.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Longueur des messages

In [4]:
length_stats = features.groupby('label')[['n_chars', 'n_words']].describe(percentiles=[.25, .5, .75])
wanted = ['mean', '50%', 'min', 'max', '25%', '75%']
display(length_stats.loc[:, (slice(None), wanted)].round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color in [('ham', '#3b82f6'), ('spam', '#ef4444')]:
    values = features.loc[features['label'] == label, 'n_chars']
    axes[0].hist(values, bins=40, alpha=.55, label=label, color=color)
axes[0].set(title='Distribution de la longueur', xlabel='Caractères', ylabel='Fréquence')
axes[0].legend()
features.boxplot(column='n_words', by='label', ax=axes[1], grid=False)
axes[1].set(title='Nombre approximatif de mots', xlabel='Label', ylabel='Mots')
fig.suptitle('')
plt.tight_layout()
plt.show()

,n_chars,n_words,n_chars,n_words,n_chars,n_words,n_chars,n_words,n_chars,n_words,n_chars,n_words
,mean,mean,50%,50%,min,min,max,max,25%,25%,75%,75%
label,,,,,,,,,,,,
ham,77.78,13.98,55.0,10.0,2.0,1.0,986.0,184.0,35.0,6.0,100.0,18.0
spam,157.74,25.03,167.0,26.0,8.0,1.0,262.0,41.0,145.0,22.0,181.0,29.0


/var/folders/vl/wsdldpb93sv0d2b1g9kn49380000gn/T/ipykernel_16084/828493159.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analyse textuelle légère
Les variables ci-dessous sont exploratoires uniquement et ne sont utilisées pour aucun entraînement.

In [5]:
for label in ['ham', 'spam']:
    display(Markdown(f'### Exemples `{label}`'))
    display(df.loc[df['label'] == label, ['id', 'text_fr']].sample(5, random_state=RANDOM_SEED))

indicator_summary = features.groupby('label').agg(
    messages_avec_url=('has_url', 'mean'),
    messages_avec_numero=('has_number', 'mean'),
    caracteres_speciaux_moyens=('n_special_chars', 'mean'),
    proportion_majuscules_moyenne=('uppercase_ratio', 'mean'),
)
indicator_summary[['messages_avec_url', 'messages_avec_numero']] *= 100
display(indicator_summary.round(3))
indicator_summary[['messages_avec_url', 'messages_avec_numero']].plot.bar(
    color=['#8b5cf6', '#f59e0b'], rot=0, ylabel='Messages concernés (%)',
    title='Présence d’URL et de nombres'
)
plt.tight_layout()
plt.show()

### Exemples `ham`

,id,text_fr
3714,sgfr-003714,"Si je ne rencontre pas tous les rites, je rent..."
1311,sgfr-001311,"Je serai toujours là, même si c'est juste dans..."
548,sgfr-000548,"Désolé que ça ait pris si longtemps, omw maint..."
1324,sgfr-001324,J'ai 50 shd être ok dit plus moins 10. Est-ce ...
3184,sgfr-003184,Dunno i juz askin parce que j'ai une carte a o...


### Exemples `spam`

,id,text_fr
1456,sgfr-001456,Summers enfin ici! Fantastique un chat ou flir...
1853,sgfr-001853,C'est la 2ème fois que nous avons essayé 2 con...
673,sgfr-000673,Get ur 1st RINGTONE GRATUITEMENT! Répondre à c...
947,sgfr-000947,Le solde de trésorerie d'Ur est actuellement d...
2881,sgfr-002881,Last Chance! Réclamez une valeur de £150 de bo...


,messages_avec_url,messages_avec_numero,caracteres_speciaux_moyens,proportion_majuscules_moyenne
label,,,,
ham,0.352,13.990,4.902,0.052
spam,16.332,94.378,6.705,0.146


/var/folders/vl/wsdldpb93sv0d2b1g9kn49380000gn/T/ipykernel_16084/3358468955.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Qualité et limites de la traduction française

L'échantillon reproductible suivant permet une comparaison humaine de 10 ham et 10 spam. Il ne constitue pas une mesure automatique de qualité linguistique. À la lecture, on recherche seulement des anomalies manifestes : formulations peu naturelles ou littérales, monnaies britanniques, numéros et références culturelles anglophones.

In [6]:
translation_sample = (
    df.groupby('label', group_keys=False)
      .sample(n=10, random_state=RANDOM_SEED)
      .reset_index(drop=True)
)
display(translation_sample[['label', 'id', 'text_en', 'text_fr']])

observable_markers = pd.Series({
    'monnaie britannique (£ ou livre)': df['text_fr'].str.contains(r'£|\blivres?\b', case=False, regex=True, na=False).sum(),
    'référence anglophone indicative': df['text_fr'].str.contains(r'£|\bpounds?\b|\buk\b|london|wimbledon|nokia|orange', case=False, regex=True, na=False).sum(),
})
display(observable_markers.to_frame('nombre de messages'))
print('Ces marqueurs orientent la lecture ; ils ne notent pas la traduction.')

,label,id,text_en,text_fr
0,ham,sgfr-003714,If i not meeting ü all rite then i'll go home ...,"Si je ne rencontre pas tous les rites, je rent..."
1,ham,sgfr-001311,"I.ll always be there, even if its just in spir...","Je serai toujours là, même si c'est juste dans..."
2,ham,sgfr-000548,"Sorry that took so long, omw now","Désolé que ça ait pris si longtemps, omw maint..."
3,ham,sgfr-001324,I thk 50 shd be ok he said plus minus 10.. Did...,J'ai 50 shd être ok dit plus moins 10. Est-ce ...
4,ham,sgfr-003184,Dunno i juz askin cos i got a card got 20% off...,Dunno i juz askin parce que j'ai une carte a o...
5,ham,sgfr-001075,Aight ill get on fb in a couple minutes,La vue mal monter sur fb dans quelques minutes
6,ham,sgfr-001412,somewhere out there beneath the pale moon ligh...,Quelque part là-bas sous la lumière de la lune...
7,ham,sgfr-003043,Slaaaaave ! Where are you ? Must I summon you ...,Slaaaave! Où es-tu? Dois-je t'invoquer tout le...
8,ham,sgfr-000014,I HAVE A DATE ON SUNDAY WITH WILL!!,J'ai un rendez-vous le dimanche avec Will!!
9,ham,sgfr-005097,Sorry about that this is my mates phone and i ...,Désolé pour ça c'est mon téléphone de potes et...


,nombre de messages
monnaie britannique (£ ou livre),288
référence anglophone indicative,363


Ces marqueurs orientent la lecture ; ils ne notent pas la traduction.


## Data leakage et doublons
Aucun split train/test n'est effectué ici. Lors de la modélisation, le split devra précéder toute opération apprenant à partir des données, et les groupes de doublons devront être gérés pour éviter leur présence dans plusieurs partitions.

In [7]:
def duplicate_report(column):
    sizes = df.groupby(column, dropna=False).size()
    conflicts = df.groupby(column, dropna=False)['label'].nunique()
    return {
        'groupes dupliqués': int((sizes > 1).sum()),
        'lignes excédentaires': int((sizes - 1).clip(lower=0).sum()),
        'textes avec labels différents': int((conflicts > 1).sum()),
    }

leakage_report = pd.DataFrame({
    'français': duplicate_report('text_fr'),
    'anglais': duplicate_report('text_en'),
})
display(leakage_report)
print(f'Doublons exacts français + anglais + label (hors id) : {exact_duplicates}')

,français,anglais
groupes dupliqués,295,289
lignes excédentaires,438,415
textes avec labels différents,0,0


Doublons exacts français + anglais + label (hors id) : 415


## Choix futurs des métriques

Le dataset est déséquilibré : l'accuracy seule peut être insuffisante. Nous suivrons notamment la **precision sur spam**, le **recall sur spam**, le **F1-score spam**, le **F1 macro** et la **matrice de confusion**.

Le compromis métier oppose deux erreurs : un **faux positif** est un SMS légitime classé spam ; un **faux négatif** est un spam non détecté. Aucune métrique prioritaire n'est décidée ici. Cette décision fera l'objet de l'ADR-001.

## Génération du rapport automatique

In [8]:
audit = run_audit()
display(pd.DataFrame(audit['label_distribution']).T)
print('Rapports écrits :')
print('-', PROJECT_ROOT / 'outputs/reports/data_audit.json')
print('-', PROJECT_ROOT / 'outputs/reports/data_audit.md')
print('Aucun modèle entraîné ; aucun split réalisé.')

,count,percentage
ham,4825.0,86.59
spam,747.0,13.41


Rapports écrits :
- outputs/reports/data_audit.json
- outputs/reports/data_audit.md
Aucun modèle entraîné ; aucun split réalisé.
